# 02 ? Auditor?a de calidad de datos

## Objetivo y alcance
Revisar los cinco CSV de microdatos registrados en `source_schema.json` y producir m?tricas agregadas reproducibles. Las fuentes se leen como texto y permanecen intactas. Esta etapa no confirma claves, no une tablas y no construye la base relacional ni el dataset anal?tico.

## M?todo
- Se usan las rutas, codificaciones, separadores y encabezados exactos del esquema preliminar. Un encabezado distinto detiene la auditor?a.
- `""` se cuenta como vac?o; una celda no vac?a compuesta solo por espacios se cuenta aparte. Tokens como `NA` y `NULL` se conservan como texto.
- El perfil num?rico indica qu? proporci?n de celdas no vac?as puede interpretarse como n?mero; no recomienda tipos de almacenamiento.
- Los duplicados exactos se reconocen en memoria mediante huellas ef?meras. No se guardan valores de filas ni huellas.
- Los identificadores candidatos proceden solo de nombres de columna. Vac?os, distintos y repeticiones son indicadores, no validaciones de clave.

## Limitaciones
Los espacios no se eliminan ni se normalizan; las categor?as no se recodifican. Las filas con ancho irregular se contabilizan, y las m?tricas por columna consideran ?nicamente posiciones definidas por el encabezado. Los reportes contienen rutas, nombres de columnas y m?tricas agregadas, nunca filas fuente.

In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontr? pyproject.toml desde el directorio actual")
src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from mode_choice.quality_audit import audit_source_csvs, write_audit_reports

# Primero se auditan todas las fuentes; solo si todas se leen correctamente se escriben reportes.
table_results, column_results = audit_source_csvs(PROJECT_ROOT)
table_report, column_report = write_audit_reports(PROJECT_ROOT, table_results, column_results)

tables = pd.DataFrame(table_results)
columns = pd.DataFrame(column_results)
print(f"Tablas auditadas: {len(tables)}")
print(f"Columnas auditadas: {len(columns)}")
print(f"Reportes: {table_report.relative_to(PROJECT_ROOT)}; {column_report.relative_to(PROJECT_ROOT)}")
display(tables)
display(columns.groupby("source_path", sort=False).agg(
    columns=("source_name", "count"),
    empty_cells=("empty_cell_count", "sum"),
    whitespace_only_cells=("whitespace_only_cell_count", "sum"),
    numeric_cells=("numeric_parseable_count", "sum"),
    identifier_candidates=("is_identifier_candidate", "sum"),
))

Tablas auditadas: 5
Columnas auditadas: 171
Reportes: docs\data_dictionary\data_quality_audit_tables.csv; docs\data_dictionary\data_quality_audit_columns.csv


,source_path,column_count,row_count,row_width_mismatch_count,exact_duplicate_row_count,empty_cell_count,whitespace_only_cell_count
0,data/sources/raw/Database/a. Modulo hogares.csv,27,22755,0,0,0,0
1,data/sources/raw/Database/b. Modulo vehiculos.csv,19,17392,0,0,17392,0
2,data/sources/raw/Database/c. Modulo personas.csv,40,67556,0,0,0,0
3,data/sources/raw/Database/d. Modulo viajes.csv,47,100174,0,0,0,0
4,data/sources/raw/Database/e. Modulo etapas.csv,38,110881,0,0,0,0


,columns,empty_cells,whitespace_only_cells,numeric_cells,identifier_candidates
source_path,,,,,
data/sources/raw/Database/a. Modulo hogares.csv,27,0,0,290566,3
data/sources/raw/Database/b. Modulo vehiculos.csv,19,17392,0,136107,4
data/sources/raw/Database/c. Modulo personas.csv,40,0,0,456152,3
data/sources/raw/Database/d. Modulo viajes.csv,47,0,0,1237627,6
data/sources/raw/Database/e. Modulo etapas.csv,38,0,0,1101478,5
